In [1]:
!pip install -q transformers==4.44.2 huggingface_hub==0.36.2 peft==0.19.1 accelerate==1.13.0 bitsandbytes jsonformer pandas openpyxl pdfminer.six python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.9 MB/s eta 0:00:00


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MAPS III 2025 Full Pipeline – 3-run Chain Evaluation + Majority Voting
"""

import os
import json
import re
import time
import logging
from typing import Dict, List, Optional, Any, Tuple
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from jsonformer import Jsonformer
import transformers
from jsonschema import Draft7Validator, validators as _js_validators
import docx
from pdfminer.high_level import extract_text
import pathlib
from collections import Counter

# ----- 导入 mapping -----
import sys
sys.path.insert(0, "/kaggle/input/datasets/zz4825/mapping-py")
import mapping

# ----- 配置（请根据实际修改）-----
EXCEL_PATH = "/kaggle/input/datasets/zz4825/v5-chain-class/100_full_pipeline_input.xlsx"          # 必须包含 GT_Chain 列
SHEET_NAME = "Sheet1"
OUTPUT_DIR = "./output"
ADAPTER_DIR = "/kaggle/input/datasets/zz4825/v5-1-5b-output/qwen1.5b_finetune_sm_ft_s43_20260722_182053/finetuned/lora_adapter"
SCHEMA_PATH = "/kaggle/input/datasets/zz4825/schema/extraction_schema_2025_sc_39new.json"
ALIAS_MAP_PATH = "/kaggle/input/datasets/zz4825/qwen-alias/qwen_output_alias_map4_1.json"
# 字段级 Ground Truth 目录（若为 None 或目录不存在，则跳过字段评估）
FIELD_GT_DIR = "/kaggle/input/datasets/zz4825/fulltest-v5-100p"

MAX_LEN_INPUT = 4096
JSONFORMER_MAX_STRING_TOKEN_LENGTH = 128

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger("MAPS3_Integrate")

# ----- 1. 加载 Schema 和 验证器（与V5完全一致）-----
def load_json_schema(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Schema file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        schema = json.load(f)
    ValidatorCls = _js_validators.validator_for(schema)
    ValidatorCls.check_schema(schema)
    validator = ValidatorCls(schema)
    return schema, validator

ENDOSCOPY_SCHEMA, ENDO_VALIDATOR = load_json_schema(SCHEMA_PATH)
REQUIRED_KEYS = list(ENDOSCOPY_SCHEMA.get("required", []))
PROPS = ENDOSCOPY_SCHEMA.get("properties", {})
ALLOWED_ENUMS = {k: v["enum"] for k, v in PROPS.items() if isinstance(v, dict) and "enum" in v}
PATTERNS = {k: v["pattern"] for k, v in PROPS.items() if isinstance(v, dict) and "pattern" in v}

# ----- 2. 别名映射（与V5完全一致）-----
def load_alias_map(path: str) -> dict:
    if not os.path.exists(path):
        print(f"[WARN] Alias map not found: {path}, using empty mapping.")
        return {"global_value_aliases": {}, "field_name_aliases": {}, "field_value_aliases": {}}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

ALIAS_MAP = load_alias_map(ALIAS_MAP_PATH)
NUMERIC_LIKE_FIELDS = {"neoplastic_lesion_size", "pepsinogen_i_level", "pepsinogen_ii_level", "patient_age"}

def apply_alias_map(value: Any, key: str) -> str:
    if not isinstance(value, str):
        value = str(value)
    original = value.strip()
    if original == "":
        return "not_mentioned"
    if key in NUMERIC_LIKE_FIELDS:
        return original
    field_aliases = ALIAS_MAP.get("field_value_aliases", {}).get(key, {})
    if original in field_aliases:
        mapped = field_aliases[original]
        return mapped if mapped is not None else "not_mentioned"
    global_aliases = ALIAS_MAP.get("global_value_aliases", {})
    if original in global_aliases:
        mapped = global_aliases[original]
        return mapped if mapped is not None else "not_mentioned"
    return original

# ----- 3. 标准化与强制转换（与V5完全一致）-----
def _norm_enum_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.strip().lower())

_NOT_MENTIONED_NORM = {_norm_enum_text(x) for x in [
    "not_mentioned", "not mentioned", "notmentioned",
    "n/a", "na", "none", "unknown", "-", "null", ""
]}
_ENUM_NORM_MAP = {k: {_norm_enum_text(v): v for v in vals} for k, vals in ALLOWED_ENUMS.items()}
_NUM_RE = re.compile(r"(\d+(?:\.\d+)?)")

def _coerce_to_allowed(key: str, value: Any) -> str:
    if value is None:
        return "not_mentioned"
    s = str(value).strip()
    if not s or _norm_enum_text(s) in _NOT_MENTIONED_NORM:
        return "not_mentioned"
    # 数值字段特殊处理
    if key == "neoplastic_lesion_size":
        m = re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm)?", s, flags=re.I)
        if not m:
            return "not_mentioned"
        val = float(m.group(1))
        unit = (m.group(2) or "mm").lower()
        if unit == "cm":
            val *= 10.0
        val_str = f"{val:.3f}".rstrip("0").rstrip(".")
        return f"{val_str} mm"
    if key == "pepsinogen_i_level":
        m = _NUM_RE.search(s)
        return f"{m.group(1)} ng/ml" if m else "not_mentioned"
    if key == "pepsinogen_ii_level":
        m = _NUM_RE.search(s)
        return f"{m.group(1)} ng/ml" if m else "not_mentioned"
    # OLGA / OLGIM
    if key in ["olga_stage", "olgim_stage"]:
        s_lower = s.lower()
        digit_map = {"0": "0", "1": "i", "2": "ii", "3": "iii", "4": "iv"}
        if s_lower in digit_map:
            return digit_map[s_lower]
        m = re.search(r'stage\s*([0-4]|i{1,3}|iv)', s_lower)
        if m:
            sub = m.group(1)
            if sub in digit_map:
                return digit_map[sub]
            if sub in ["i", "ii", "iii", "iv"]:
                return sub
        m = re.search(r'\b(i{1,3}|iv)\b', s_lower)
        if m:
            return m.group(1)
        return "not_mentioned"
    # 枚举字段
    if key in ALLOWED_ENUMS:
        if s in ALLOWED_ENUMS[key]:
            return s
        norm = _norm_enum_text(s)
        mapped = _ENUM_NORM_MAP[key].get(norm)
        return mapped if mapped is not None else "not_mentioned"
    # pattern
    if key in PATTERNS:
        return s if re.match(PATTERNS[key], s) else "not_mentioned"
    return s[:128] if len(s) > 128 else s

def validate_and_sanitize(obj: Dict[str, Any], validator: Draft7Validator) -> Dict[str, str]:
    if isinstance(obj, dict):
        field_aliases = ALIAS_MAP.get("field_name_aliases", {})
        renamed = {}
        for k, v in obj.items():
            target_key = field_aliases.get(k, k)
            renamed[target_key] = v
        obj = renamed
    cleaned = {k: "not_mentioned" for k in REQUIRED_KEYS}
    if isinstance(obj, dict):
        for k in REQUIRED_KEYS:
            if k in obj:
                raw_val = obj[k]
                aliased_val = apply_alias_map(raw_val, k)
                cleaned[k] = _coerce_to_allowed(k, aliased_val)
    for error in validator.iter_errors(cleaned):
        path_key = list(error.path)[0] if error.path else None
        if path_key in cleaned:
            cleaned[path_key] = "not_mentioned"
    return cleaned

# ----- 4. 报告文件读取（与V5一致）-----
def _read_txt_like(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def _read_pdf(path: str) -> str:
    try:
        return extract_text(path) or ""
    except Exception as e:
        logger.warning(f"PDF extract failed for {path}: {e}")
        return ""

def _read_docx(path: str) -> str:
    try:
        doc = docx.Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    except Exception as e:
        logger.warning(f"DOCX read failed for {path}: {e}")
        return ""

def read_report_file(path: Optional[str]) -> str:
    if not path:
        return ""
    ext = pathlib.Path(path).suffix.lower()
    if ext in {".txt", ".md"}:
        content = _read_txt_like(path)
    elif ext == ".pdf":
        content = _read_pdf(path)
    elif ext == ".docx":
        content = _read_docx(path)
    else:
        try:
            content = _read_txt_like(path)
        except Exception:
            return ""
    lines = content.splitlines()
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith("==="):
            continue
        if "(Correct)" in stripped or "(Wrong)" in stripped:
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)

# ----- 5. 模型加载（与V5一致）-----
def load_model():
    cfg = PeftConfig.from_pretrained(ADAPTER_DIR)
    base_model_name = cfg.base_model_name_or_path
    bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )
    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True, trust_remote_code=True)
    base.resize_token_embeddings(len(tok))
    model = PeftModel.from_pretrained(base, ADAPTER_DIR).eval()
    tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return tok, model

# ----- 6. 提取函数（Jsonformer 参数与V5一致）-----
def build_schema_guide():
    guide = {}
    for k in REQUIRED_KEYS:
        if k in ALLOWED_ENUMS:
            guide[k] = ALLOWED_ENUMS[k]
        elif k in PATTERNS:
            guide[k] = "pattern: " + PATTERNS[k]
        else:
            guide[k] = "short text / Not Mentioned"
    return guide

SCHEMA_GUIDE = build_schema_guide()
COMBINED_SINGLE_SCHEMA = {
    "type": "object",
    "properties": {
        "patient_id": {"type": "string"},
        "variant": {"type": "string", "enum": ["correct", "wrong"]},
        "endoscopy and pathology": ENDOSCOPY_SCHEMA,
    },
    "required": ["patient_id", "variant", "endoscopy and pathology"],
    "additionalProperties": False,
}

def make_joint_prompt(pid: str, variant: str, endo_text: str, path_text: str) -> str:
    return f"""
You are given two reports for the SAME patient: an Endoscopy report and a Pathology report.
Extract ONE unified set of structured findings using BOTH reports as evidence.

Return ONLY a single JSON object with this exact structure:
{{
  "patient_id": "{pid}",
  "variant": "{variant}",
  "endoscopy and pathology": {{ ... EXACT keys below ... }}
}}

Rules:
- Use EXACT spellings for keys and allowed values.
- If a finding is not reported in either report, set "not_mentioned.
- No explanations. No extra keys.

Keys and allowed values (loaded from JSON schema file):
{json.dumps(SCHEMA_GUIDE, indent=2)}

--- ENDOSCOPY REPORT ---
{endo_text}

--- PATHOLOGY REPORT ---
{path_text}
""".strip()

def extract_single(endo: str, patho: str, patient_id: str = "auto", variant: str = "auto") -> dict:
    tok, model = load_model()
    prompt = make_joint_prompt(patient_id, variant, endo, patho)
    ids = tok(prompt, add_special_tokens=False)["input_ids"]
    if len(ids) > MAX_LEN_INPUT:
        head = min(512, MAX_LEN_INPUT // 3)
        tail = MAX_LEN_INPUT - head
        ids_trunc = ids[:head] + ids[-tail:]
        prompt = tok.decode(ids_trunc, skip_special_tokens=False)
    jf = Jsonformer(
        model=model,
        tokenizer=tok,
        json_schema=COMBINED_SINGLE_SCHEMA,
        prompt=prompt,
        max_string_token_length=JSONFORMER_MAX_STRING_TOKEN_LENGTH,
        temperature=1e-6,
    )
    raw = jf()
    inner = raw.get("endoscopy and pathology", {})
    cleaned = validate_and_sanitize(inner, ENDO_VALIDATOR)
    return cleaned

def extract_with_loop(endo: str, patho: str, max_it: int = 3,
                      pid: str = "auto", var: str = "auto") -> Tuple[dict, dict]:
    history = []
    total_errors = 0
    data = {}
    for it in range(1, max_it + 2):
        t0 = time.time()
        data = extract_single(endo, patho, pid, var)
        elapsed = time.time() - t0
        errors = [k for k in REQUIRED_KEYS if data.get(k) is None]
        total_errors = len(errors)
        history.append({"iteration": it, "errors": total_errors, "time_s": elapsed})
        if not errors:
            logger.debug(f"Extraction succeeded in {it} tries")
            return data, {"iterations": it, "errors": 0, "history": history}
        logger.info(f"Extraction attempt {it}: {total_errors} missing keys, retrying...")
    return data, {"iterations": max_it, "errors": total_errors, "history": history}

# ----- 7. 单轮处理函数（返回记录 DataFrame）-----
def process_excel(excel_path: str, sheet_name: str, output_dir: str, round_seed: int):
    torch.manual_seed(round_seed)
    np.random.seed(round_seed)

    round_out = os.path.join(output_dir, f"round_{round_seed}")
    pred_dir = os.path.join(round_out, "predictions")
    os.makedirs(pred_dir, exist_ok=True)

    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    if "patient_id" not in df.columns:
        df.insert(0, "patient_id", [f"row_{i}" for i in range(len(df))])

    has_gt = "GT_Chain" in df.columns
    if not has_gt:
        logger.warning("Excel does not contain 'GT_Chain' column; chain accuracy will not be evaluated.")

    records = []          # 链级记录
    field_records = []    # 字段级记录

    for idx, row in df.iterrows():
        patho_raw = row.get("Pathology Report", "")
        patho = read_report_file(patho_raw) if isinstance(patho_raw, str) and os.path.exists(patho_raw) else str(patho_raw)
        row_num = idx + 1

        for endo_col, variant_num in [
            ("Endoscopy Report 1", 1),
            ("Endoscopy Report 2", 2)
        ]:
            if endo_col not in row or pd.isna(row[endo_col]) or str(row[endo_col]).strip() == "":
                continue
            endo_raw = str(row[endo_col])
            endo = read_report_file(endo_raw) if os.path.exists(endo_raw) else endo_raw
            pid = f"Patient_{row_num}-{variant_num}"
            # 尝试从 Excel 的 "patient_id" 列读取 GT ID
            gt_id = row.get("patient_id")
            if pd.isna(gt_id):
                # 若没有，则使用行号生成
                gt_id = f"Patient_{row_num}-{variant_num}"
            pid = gt_id  # 用于输出文件命名，保持一致
            variant_str = "correct"

            logger.info(f"[Round {round_seed}] Extracting {pid} ...")
            data, loop_info = extract_with_loop(endo, patho, pid=pid, var=variant_str)

            result = mapping.classify_patient(data)
            chain_str = result.get("chain")
            if chain_str is None:
                chain_str = "N/A"
            status = result.get("status")
            surv_text = result.get("surveillance_text") or "Unknown"

            # 决策记录
            decision_record = {
                "patient_id": pid,
                "variant": endo_col,
                "Chain": chain_str if chain_str != "N/A" else ["N/A"],
                "Surveillance_Time": surv_text,
                "status": status,
                "warnings": result.get("warnings", []),
                "treatment_suggestion": result.get("treatment_suggestion"),
                "hp_eradication_suggested": result.get("hp_eradication_suggested", False),
                "missing_fields": result.get("missing_fields", []),
                "advanced_olg_staging": result.get("advanced_olg_staging", False),
                "extraction_iterations": loop_info["iterations"],
                "extraction_errors": loop_info["errors"]
            }
            human_readable = mapping.format_final_output(result, data)
            decision_record["human_readable"] = human_readable

            out_json = os.path.join(pred_dir, f"{pid}_{endo_col}_decision.json")
            with open(out_json, "w") as f:
                json.dump(decision_record, f, indent=2)

            pred_json = {"patient_id": pid, "variant": endo_col, "endoscopy and pathology": data}
            with open(os.path.join(pred_dir, f"{pid}_{endo_col}_extracted.json"), "w") as f:
                json.dump(pred_json, f, indent=2)

            # 链级 GT
            gt_chain = row.get("GT_Chain", "") if has_gt else None
            if isinstance(gt_chain, float) and gt_chain == int(gt_chain):
                gt_chain = str(int(gt_chain))
            else:
                gt_chain = str(gt_chain).strip() if pd.notna(gt_chain) else ""

            records.append({
                "patient_id": pid,
                "variant": endo_col,
                "pred_chain": chain_str,
                "gt_chain": gt_chain if has_gt else None,
                "status": status,
                "warnings": "; ".join(result.get("warnings", [])),
                "surveillance_text": surv_text,
                "treatment_suggestion": result.get("treatment_suggestion"),
                "hp_eradication_suggested": result.get("hp_eradication_suggested", False),
                "smoking_cessation_suggested": (data.get("smoking_status") == "smoker"),
                "advanced_olg_staging": result.get("advanced_olg_staging", False),
                "human_readable": human_readable  # <-- 输出文字建议
            })

            # 字段级 GT 比较
            if FIELD_GT_DIR and os.path.isdir(FIELD_GT_DIR):
                gt_filename = f"{pid}_extracted_correct.json"
                gt_path = os.path.join(FIELD_GT_DIR, gt_filename)
                if os.path.exists(gt_path):
                    try:
                        with open(gt_path, "r", encoding="utf-8") as f:
                            gt_json = json.load(f)
                        gt_flat = gt_json.get("endoscopy and pathology", gt_json)
                        field_correct = {}
                        for field in REQUIRED_KEYS:
                            gt_val = str(gt_flat.get(field, "not_mentioned"))
                            pred_val = str(data.get(field, "not_mentioned"))
                            field_correct[field] = (gt_val == pred_val)
                        field_correct["patient_id"] = pid
                        field_correct["variant"] = endo_col
                        field_records.append(field_correct)
                    except Exception as e:
                        logger.warning(f"Failed to load or compare GT for {gt_path}: {e}")
                else:
                    if not hasattr(process_excel, "_gt_missing_logged"):
                        process_excel._gt_missing_logged = set()
                    if gt_path not in process_excel._gt_missing_logged:
                        logger.warning(f"Field GT file not found: {gt_path}")
                        process_excel._gt_missing_logged.add(gt_path)

    # 保存链记录
    records_df = pd.DataFrame(records)
    records_df.to_csv(os.path.join(round_out, "round_records.csv"), index=False)

    # 保存字段比较结果并返回字段准确率
    field_acc_series = None
    if field_records:
        field_df = pd.DataFrame(field_records)
        field_df.to_csv(os.path.join(round_out, "field_comparison.csv"), index=False)
        # 计算每个字段的准确率（仅计算 REQUIRED_KEYS 中的字段）
        field_acc = field_df[[f for f in REQUIRED_KEYS if f in field_df.columns]].mean()
        field_acc.to_csv(os.path.join(round_out, "field_accuracy.csv"))
        logger.info(f"Round {round_seed} field-level accuracy (avg over {len(field_records)} samples): {field_acc.mean():.4f}")
        field_acc_series = field_acc  # 返回 Series

    logger.info(f"Round {round_seed} completed. Saved {len(records_df)} records.")
    return records_df, field_acc_series

# ----- 8. 主程序：3轮运行 + 评估 + 众数投票 -----
if __name__ == "__main__":
    # ----- 辅助函数：标准化链（忽略顺序） -----
    def normalize_chain(chain_str: str) -> str:
        if chain_str in ["N/A", "Ambiguous", None]:
            return chain_str if chain_str is not None else "N/A"
        parts = [c.strip() for c in str(chain_str).replace(" ", "").split(",") if c.strip()]
        return ",".join(sorted(parts)) if parts else "N/A"

    def chains_equal(chain1, chain2) -> bool:
        return normalize_chain(chain1) == normalize_chain(chain2)

    def jaccard_similarity(chain1, chain2) -> float:
        set1 = set(c.strip() for c in normalize_chain(chain1).split(",") if c.strip() and c.strip() != "N/A")
        set2 = set(c.strip() for c in normalize_chain(chain2).split(",") if c.strip() and c.strip() != "N/A")
        if not set1 and not set2:
            return 1.0
        inter = len(set1 & set2)
        union = len(set1 | set2)
        return inter / union if union > 0 else 0.0

    # ----- 单轮运行（固定种子 42） -----
    SEED = 42
    df_round, field_acc = process_excel(EXCEL_PATH, SHEET_NAME, OUTPUT_DIR, round_seed=SEED)

    # ----- 链指标评估（Perfect Match & Jaccard） -----
    valid = df_round[df_round['gt_chain'].notna() & (df_round['gt_chain'] != "")]
    if len(valid) > 0:
        perfect_acc = valid.apply(
            lambda row: chains_equal(row['pred_chain'], row['gt_chain']),
            axis=1
        ).mean()
        jaccard_mean = valid.apply(
            lambda row: jaccard_similarity(row['pred_chain'], row['gt_chain']),
            axis=1
        ).mean()
    else:
        perfect_acc = np.nan
        jaccard_mean = np.nan

    logger.info(f"Perfect Match Accuracy: {perfect_acc:.4f} (n={len(valid)})")
    logger.info(f"Mean Jaccard Similarity: {jaccard_mean:.4f} (n={len(valid)})")

    # 保存链指标汇总（单轮）
    summary_chain = pd.DataFrame({
        'Round': [SEED],
        'Perfect_Match_Accuracy': [perfect_acc],
        'Jaccard_Similarity': [jaccard_mean]
    })
    summary_chain.to_csv(os.path.join(OUTPUT_DIR, "chain_accuracy_summary.csv"), index=False)
    print("\n===== Chain Accuracy Summary =====")
    print(summary_chain.to_string(index=False))

    # 保存所有预测记录（含建议）
    # 将 human_readable 中的换行符替换为空格，方便在 Excel 中查看
    df_display = df_round.copy()
    if 'human_readable' in df_display.columns:
        df_display['human_readable'] = df_display['human_readable'].str.replace('\n', ' ', regex=False)
    df_display.to_csv(os.path.join(OUTPUT_DIR, "all_predictions_with_advice.csv"), index=False)
    logger.info(f"Saved all predictions with advice to all_predictions_with_advice.csv")

    # ----- 字段准确率汇总 -----
    if field_acc is not None:
        field_summary = pd.DataFrame({
            'Field': field_acc.index,
            'Accuracy': field_acc.values
        })
        field_summary.to_csv(os.path.join(OUTPUT_DIR, "field_accuracy_summary.csv"), index=False)
        print("\n===== Field Accuracy Summary =====")
        print(field_summary.to_string(index=False))
        print(f"\nOverall field extraction accuracy: {field_acc.mean():.4f}")

    # ----- Chain 频次分布表（基于当前轮预测结果） -----
    def parse_chains(chain_str):
        if chain_str in ["N/A", "Ambiguous", None]:
            return []
        parts = [p.strip() for p in str(chain_str).replace(" ", "").split(",") if p.strip().isdigit()]
        return [int(p) for p in parts]

    pred_counts = {}
    gt_counts = {}
    for _, row in df_round.iterrows():
        pred_chain = row.get("pred_chain")
        gt_chain = row.get("gt_chain")
        for c in parse_chains(pred_chain):
            pred_counts[c] = pred_counts.get(c, 0) + 1
        for c in parse_chains(gt_chain):
            gt_counts[c] = gt_counts.get(c, 0) + 1

    all_ids = list(range(1, 22))
    pred_series = pd.Series({i: pred_counts.get(i, 0) for i in all_ids})
    gt_series = pd.Series({i: gt_counts.get(i, 0) for i in all_ids})

    freq_df = pd.DataFrame({
        'Chain': all_ids,
        'Predicted_Count': pred_series.values,
        'GT_Count': gt_series.values,
        'Difference': pred_series.values - gt_series.values
    })
    freq_df.to_csv(os.path.join(OUTPUT_DIR, "chain_frequency_comparison.csv"), index=False)
    print("\n===== Chain Frequency Distribution =====")
    print(freq_df.to_string(index=False))

    # 绘制柱状图并添加数值标签
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(14, 6))
        x = np.arange(len(all_ids))
        width = 0.35
        bars1 = plt.bar(x - width/2, pred_series.values, width, label='Predicted', color='skyblue')
        bars2 = plt.bar(x + width/2, gt_series.values, width, label='Ground Truth', color='salmon')
        plt.xlabel('Chain ID')
        plt.ylabel('Frequency')
        plt.title('Chain Frequency Distribution (Predicted vs Ground Truth)')
        plt.xticks(x, all_ids)
        plt.legend()
        # 添加数值标签
        for bar in bars1:
            height = bar.get_height()
            if height > 0:
                plt.text(bar.get_x() + bar.get_width()/2., height + 0.2, f'{int(height)}', ha='center', va='bottom', fontsize=9)
        for bar in bars2:
            height = bar.get_height()
            if height > 0:
                plt.text(bar.get_x() + bar.get_width()/2., height + 0.2, f'{int(height)}', ha='center', va='bottom', fontsize=9)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "chain_frequency_comparison.png"), dpi=150)
        plt.close()
        print(f"Saved frequency plot to {os.path.join(OUTPUT_DIR, 'chain_frequency_comparison.png')}")
    except ImportError:
        print("matplotlib not available, skipping plot.")

    print("\nAll done. Results saved in", OUTPUT_DIR)

2026-08-08 23:24:57,966 INFO [Round 42] Extracting AG_Patient_001 ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
2026-08-08 23:27:24,042 INFO [Round 42] Extracting AG_Patient_002 ...
2026-08-08 23:28:47,961 INFO [Round 42] Extracting AG_Patient_003 ...
2026-08-08 23:30:09,988 INFO [Round 42] Extracting AG_Patient_004 ...
2026-08-08 23:31:29,250 INFO [Round 42] Extracting AG_Patient_005 ...
2026-08-08 23:32:51,144 INFO [Round 42] Extracting AG_Patient_006 ...
2026-08-08 23:34:12,632 INFO [Round 42] Extracting AG_Patient_007 ...
2026-08-08 23:35:28,647 INFO [Round 42] Extracting AG_Patient_008 ...
2026-08-08 23:36:48,667 INFO [Round 42] Extracting AG_Patient_009 ...
2026-08-08 23:38:08,448 INFO [Round 42] Extracting AG_Patient_010 ...
2026-08-08 23:39:27,459 INFO [Round 42] Extracting AG_Patient_011 ...
2026-08-08 23:40:47,996 INFO [Round 42] Extracting AG_Patient_012 ...
2


===== Chain Accuracy Summary =====
 Round  Perfect_Match_Accuracy  Jaccard_Similarity
    42                    0.97                0.97

===== Field Accuracy Summary =====
                           Field  Accuracy
  kimura_takemoto_classification      1.00
                     eggim_score      1.00
                      olga_stage      1.00
                     olgim_stage      1.00
      helicobacter_pylori_status      1.00
              atrophic_gastritis      1.00
incomplete_intestinal_metaplasia      0.97
                  family_history      0.99
                       dysplasia      0.96
                 dysplasia_grade      0.95
            dysplasia_visibility      0.94
        indefinite_for_dysplasia      0.90
    neoplastic_lesion_visibility      1.00
               neoplastic_lesion      0.99
     lesion_paris_classification      0.97
          neoplastic_lesion_size      1.00
    neoplastic_lesion_ulceration      0.99
          differentiation_status      1.00
         